# M1 baseline — CVE severity from the description (#7)

**Question (FR-23, risk R4):** can the text of a CVE description predict its CVSS severity class well enough to justify the final model M1 in Sprint 3?

| CRISP-DM step | In this notebook |
|---|---|
| Business understanding | Since April 2026 many CVEs get no NVD score; M1 must estimate their severity |
| Data understanding | CVEs loaded by the pipeline (`run_pipeline --source cve`) |
| Data preparation | Label = CVSS **3.x** severity (CNA, else CISA ADP); feature = description text |
| Modeling | Majority-class baseline, then TF-IDF + logistic regression (and a linear SVM for comparison) |
| Evaluation | **Time-based split**, macro-F1, per-class F1, confusion matrix |
| Decision | Go / no-go for M1 |

Prerequisites: Docker running (`docker compose up -d`), CVEs loaded, virtual environment `.venv` with `requirements-dev.txt`.

## 0. Settings

In [1]:
# Time-based split: train on older CVEs, test on the most recent period (never a random split)
TRAIN_FROM = "2018-01-01"   # earlier records have less reliable dates (risk R3)
TEST_FROM = "2025-07-01"    # most recent period = test set
RANDOM_STATE = 42           # fixed seed (reproducibility)
TARGET_MACRO_F1 = 0.70      # indicative target of FR-23

## 1. Connection to the operational database

Credentials are read from the `.env` file at the repository root (never written in the notebook).

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg

def find_repo_root(start=Path.cwd()):
    for folder in [start, *start.parents]:
        if (folder / "docker-compose.yml").exists():
            return folder
    raise FileNotFoundError("Run the notebook from inside the threatlens repository")

def read_env(path):
    values = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            values[key.strip()] = value.strip()
    return values

ROOT = find_repo_root()
env = read_env(ROOT / ".env")
conn = psycopg.connect(
    host="localhost",  # the notebook runs on the laptop, PostgreSQL is published on localhost:5432
    port=int(env.get("POSTGRES_PORT", 5432)),
    dbname=env["POSTGRES_DB"],
    user=env["POSTGRES_USER"],
    password=env["POSTGRES_PASSWORD"],
)

def query(sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        columns = [c.name for c in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=columns)

print("Connected to", env["POSTGRES_DB"])

Connected to threatlens


## 2. Data understanding

In [3]:
overview = query("""
    SELECT
        COUNT(*)                                                      AS cves,
        COUNT(*) FILTER (WHERE NOT EXISTS (
            SELECT 1 FROM cti_cvssmetric m WHERE m.vulnerability_id = v.id))   AS without_any_score,
        COUNT(*) FILTER (WHERE EXISTS (
            SELECT 1 FROM cti_cvssmetric m WHERE m.vulnerability_id = v.id
              AND m.version IN ('3.0', '3.1')))                         AS with_cvss3
    FROM cti_vulnerability v
""")
overview["share_without_score"] = (overview.without_any_score / overview.cves).round(3)
overview

,cves,without_any_score,with_cvss3,share_without_score
0,391772,172830,203469,0.441


In [4]:
scores = query("""
    SELECT m.provider, m.version, COUNT(*) AS n
    FROM cti_cvssmetric m
    GROUP BY m.provider, m.version
    ORDER BY m.provider, m.version
""")
scores.pivot(index="provider", columns="version", values="n").fillna(0).astype(int)

version,2.0,3.0,3.1,4.0
provider,,,,
CISA_ADP,0,0,44090,4
CNA,16812,31740,145979,42476


## 3. Data preparation

One label per CVE, **CVSS 3.x only** (no mixing of v2 / v4 scales). Priority when several exist: CNA 3.1 > CNA 3.0 > CISA ADP 3.1 > CISA ADP 3.0. The class NONE (score 0) is too rare to learn and is left out.

In [5]:
raw = query("""
    SELECT v.cve_id, v.description, v.published_at, m.provider, m.version, m.severity
    FROM cti_vulnerability v
    JOIN cti_cvssmetric m ON m.vulnerability_id = v.id
    WHERE m.version IN ('3.0', '3.1') AND m.provider IN ('CNA', 'CISA_ADP')
""")
priority = {("CNA", "3.1"): 0, ("CNA", "3.0"): 1, ("CISA_ADP", "3.1"): 2, ("CISA_ADP", "3.0"): 3}
raw["rank"] = [priority[(p, v)] for p, v in zip(raw.provider, raw.version)]
data = (raw.sort_values("rank").drop_duplicates("cve_id")
           .query("severity != 'NONE'")
           .dropna(subset=["published_at"])
           .reset_index(drop=True))
data["published_at"] = pd.to_datetime(data.published_at, utc=True)
data = data[data.published_at >= pd.Timestamp(TRAIN_FROM, tz="UTC")]
print(f"{len(data):,} labelled CVEs from {TRAIN_FROM}")
data.groupby(data.published_at.dt.year).severity.value_counts().unstack(fill_value=0)

202,325 labelled CVEs from 2018-01-01


severity,CRITICAL,HIGH,LOW,MEDIUM
published_at,,,,
2018,175,638,104,827
2019,224,1040,110,1268
2020,505,2395,313,2867
2021,761,3577,491,3555
2022,1668,5682,936,6486
2023,2350,8129,1335,11335
2024,3614,13729,1753,17628
2025,3316,15754,2113,19525
2026,7874,30087,2926,27235


In [6]:
cutoff = pd.Timestamp(TEST_FROM, tz="UTC")
train = data[data.published_at < cutoff]
test = data[data.published_at >= cutoff]
CLASSES = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
split = pd.DataFrame({
    "train": train.severity.value_counts(normalize=True).reindex(CLASSES).round(3),
    "test": test.severity.value_counts(normalize=True).reindex(CLASSES).round(3),
})
print(f"train: {len(train):,} CVEs before {TEST_FROM} | test: {len(test):,} CVEs from {TEST_FROM}")
split

train: 114,569 CVEs before 2025-07-01 | test: 87,756 CVEs from 2025-07-01


,train,test
severity,,
LOW,0.053,0.045
MEDIUM,0.473,0.416
HIGH,0.377,0.431
CRITICAL,0.096,0.108


## 4. Modeling

In [7]:
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

def tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200_000,
                           sublinear_tf=True, strip_accents="unicode")

models = {
    "majority class (baseline)": DummyClassifier(strategy="most_frequent"),
    "TF-IDF + logistic regression": make_pipeline(
        tfidf(), LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
    "TF-IDF + linear SVM": make_pipeline(
        tfidf(), LinearSVC(class_weight="balanced", random_state=RANDOM_STATE)),
}

results, predictions = [], {}
for name, model in models.items():
    model.fit(train.description, train.severity)
    pred = model.predict(test.description)
    predictions[name] = pred
    results.append({"model": name,
                    "macro_F1": round(f1_score(test.severity, pred, average="macro"), 3),
                    "accuracy": round((pred == test.severity).mean(), 3)})
results = pd.DataFrame(results).set_index("model")
results

,macro_F1,accuracy
model,,
majority class (baseline),0.147,0.416
TF-IDF + logistic regression,0.516,0.591
TF-IDF + linear SVM,0.514,0.612


## 5. Evaluation of the best model

In [8]:
best = results.drop(index="majority class (baseline)").macro_F1.idxmax()
print("Best model:", best, "\n")
print(classification_report(test.severity, predictions[best], labels=CLASSES, digits=3))

Best model: TF-IDF + logistic regression 

              precision    recall  f1-score   support

         LOW      0.299     0.437     0.355      3971
      MEDIUM      0.691     0.626     0.657     36507
        HIGH      0.670     0.569     0.615     37837
    CRITICAL      0.342     0.607     0.438      9441

    accuracy                          0.591     87756
   macro avg      0.501     0.560     0.516     87756
weighted avg      0.627     0.591     0.602     87756



In [9]:
matrix = pd.DataFrame(confusion_matrix(test.severity, predictions[best], labels=CLASSES),
                      index=[f"true {c}" for c in CLASSES], columns=[f"pred {c}" for c in CLASSES])
matrix

,pred LOW,pred MEDIUM,pred HIGH,pred CRITICAL
true LOW,1734,1417,558,262
true MEDIUM,2935,22871,7330,3371
true HIGH,983,7945,21528,7381
true CRITICAL,142,871,2700,5728


## 6. Go / no-go decision

Rule agreed for the baseline: **GO** if macro-F1 ≥ 0.70 (FR-23 target); **GO with reservations** between 0.60 and 0.70 (the final M1 in Sprint 3 adds metadata features and tuning); **NO-GO** below 0.60 → plan of risk R4 (M1 limited to severity, CWE prediction dropped, ML presented as an honest limit).

In [10]:
import json
from datetime import date

best_f1 = results.loc[best, "macro_F1"]
baseline_f1 = results.loc["majority class (baseline)", "macro_F1"]
decision = "GO" if best_f1 >= TARGET_MACRO_F1 else "GO with reservations" if best_f1 >= 0.60 else "NO-GO"
summary = {
    "date": date.today().isoformat(),
    "train_period": f"{TRAIN_FROM} to {TEST_FROM}", "test_period": f"from {TEST_FROM}",
    "train_size": int(len(train)), "test_size": int(len(test)),
    "best_model": best, "macro_f1": float(best_f1), "baseline_macro_f1": float(baseline_f1),
    "decision": decision,
}
(ROOT / "ml" / "models").mkdir(parents=True, exist_ok=True)
(ROOT / "ml" / "models" / "m1_baseline_metrics.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

{
  "date": "2026-10-20",
  "train_period": "2018-01-01 to 2025-07-01",
  "test_period": "from 2025-07-01",
  "train_size": 114569,
  "test_size": 87756,
  "best_model": "TF-IDF + logistic regression",
  "macro_f1": 0.516,
  "baseline_macro_f1": 0.147,
  "decision": "NO-GO"
}


**Interpretation.** On the most recent period (Jul 2025 – Oct 2026, 87,756 CVEs), TF-IDF + logistic regression reaches **macro-F1 0.516** (accuracy 0.591), 3.5× the majority baseline (0.147) but below the indicative target of 0.70 (FR-23).

- MEDIUM (F1 0.66) and HIGH (0.62) are learned; LOW (0.36) and CRITICAL (0.44) are weak.
- Errors are mostly between neighbouring classes (7,945 HIGH → MEDIUM, 7,381 HIGH → CRITICAL, 7,330 MEDIUM → HIGH); distant errors are rare (142 CRITICAL → LOW).

**Decision: NO-GO for a text-only M1; risk R4 plan applied.** M1 keeps severity only (no CWE target). Sprint 3 adds metadata features (CNA, vendor/product, CWE once loaded) and tuning. The revised target, and a possible 3-class grouping, are to be validated with the supervisor at Validation 1. Predicted severities stay labelled as predicted (FR-06).

In [11]:
conn.close()